In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from pinecone import ServerlessSpec, Pinecone
from langchain_community.retrievers import PineconeHybridSearchRetriever
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder


In [ ]:
# Load environment variables searching parent directories
load_dotenv(find_dotenv())
api_key = os.getenv('PINECONE_API_KEY')

In [17]:
import time
pc = Pinecone(api_key=api_key)
index_name = "hybrid-search"
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    # Wait for serverless index to be fully provisioned and active
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)


In [18]:
index = pc.Index(index_name)

In [19]:
hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN not found in environment variables. Ensure your .env file is loaded and contains HF_TOKEN.")
os.environ["HF_TOKEN"] = hf_token

emmbed = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
emmbed

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4534.72it/s]


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [20]:
bm25 = BM25Encoder().default()
bm25

In [21]:
sentences = [
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans"
]

In [22]:
bm25.fit(sentences)
bm25.dump("bm25_values.json")
bm25 = BM25Encoder().load("bm25_values.json")

100%|██████████| 3/3 [00:00<00:00, 2975.39it/s]


In [23]:
ret = PineconeHybridSearchRetriever(embeddings=emmbed, sparse_encoder=bm25, index=index)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11290.04it/s]


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [25]:
# 1. Prepare a list to hold your formatted records
records = []

# 2. Iterate through your sentences and encode them manually
for i, text in enumerate(sentences):
    # Generate the dense and sparse vectors
    dense_vec = emmbed.embed_query(text)
    sparse_vec = bm25.encode_documents(text)
    
    # Format the record. Note: Langchain expects the original text to be in a metadata key named "context"
    records.append({
        "id": f"doc_{i}",
        "values": dense_vec,
        "sparse_values": sparse_vec,
        "metadata": {"context": text} 
    })

# 3. Upsert directly to Pinecone using the mandatory 'vectors=' keyword argument
index.upsert(vectors=records)

print(f"Successfully upserted {len(records)} documents!")

Successfully upserted 3 documents!


In [27]:
ret.invoke("which city i visit most")

[Document(metadata={'score': 0.216682971}, page_content='In 2021, I visited New Orleans'),
 Document(metadata={'score': 0.188894719}, page_content='In 2023, I visited Paris'),
 Document(metadata={'score': 0.187841}, page_content='In 2022, I visited New York')]